# 018-segfix-correction — analysis

# Imports

In [ ]:
import pandas as pd

from uavloc.matching.benchmark import paired_frame_bootstrap
from uavloc.track import experiment_dir, load_runs

# Params

In [ ]:
EXP = "018-segfix-correction"
BASELINE = "018.1"
REFERENCE_007_1_M = 5.861
REPRODUCE_TOL_M = 0.05
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
BAR_GAIN_M = 0.5
GUARD_MAX_LOSS_M = 0.5
N_BOOTSTRAP = 10_000
CI_PCT = 95
SEED = 42

# Data

In [ ]:
runs = {r["run"]: r for r in load_runs(EXP) if r["status"] == "ok"}
records = {run: pd.read_csv(experiment_dir(EXP) / "artifacts" / f"records-{run}.csv") for run in runs}
list(runs)

In [ ]:
def describe(run: dict) -> str:
    c = run["config"]
    if c["refiner"] == "none":
        return "LR-ASPP"
    return f"SegFix {c['segfix_checkpoint'].rsplit('/', 1)[-1].removesuffix('.pt')}, s = {c['offset_px']}"

# Run table

In [ ]:
table = pd.DataFrame({run: {"setup": describe(r),
                            **{f"@{n:g} m": r["metrics"][f"mean_position_error_m_at_{n:g}m"] for n in NOISE_M},
                            **{f"{s} @10": r["metrics"][f"mean_position_error_m_{s}"] for s in ("bushcha", "birky")},
                            "harm @10": r["metrics"]["harm_share_at_10m"],
                            "ms/corr (cpu)": r["metrics"]["ms_per_correction"]}
                      for run, r in runs.items()}).T
table.round(3)

In [ ]:
{"018.1 @10 m": runs[BASELINE]["metrics"]["mean_position_error_m_at_10m"], "007.1 @10 m": REFERENCE_007_1_M,
 "reproduces": abs(runs[BASELINE]["metrics"]["mean_position_error_m_at_10m"] - REFERENCE_007_1_M) <= REPRODUCE_TOL_M}

# Paired bootstrap

In [ ]:
paired = pd.DataFrame([{"run": run, "noise_m": n,
                        **paired_frame_bootstrap(records[run], records[BASELINE], n, N_BOOTSTRAP, CI_PCT, SEED)}
                       for run in runs if run != BASELINE for n in NOISE_M])
paired.round(3)

# Verdict

In [ ]:
def verdict(run: str) -> dict[str, bool]:
    rows = paired[paired["run"] == run].set_index("noise_m")
    primary = rows.loc[PRIMARY_NOISE_M]
    return {"bar": bool(primary["diff_m"] <= -BAR_GAIN_M and primary["ci_hi"] < 0),
            "guard": bool((rows.drop(PRIMARY_NOISE_M)["ci_lo"] <= GUARD_MAX_LOSS_M).all())}

In [ ]:
{run: verdict(run) for run in runs if run != BASELINE}